# CoverAI — Exploratory Data Analysis

This notebook explores the synthetic interaction dataset used to train the recommendation model.
Run `python scripts/train.py --fresh` to regenerate the data before running this notebook.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

plt.style.use('dark_background')
sns.set_palette('viridis')
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries loaded')

In [ ]:
# Generate or load dataset
from ml.training.data_generator import generate_interaction_dataset, save_dataset

raw_path = Path('../data/raw/interactions.parquet')
if raw_path.exists():
    df = pd.read_parquet(raw_path)
    print(f'Loaded cached dataset: {len(df):,} rows')
else:
    df = generate_interaction_dataset(n_users=5000)
    save_dataset(df, raw_path)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 1. Dataset Overview

In [ ]:
print('=== Dataset Summary ===')
print(f'Total interactions:   {len(df):,}')
print(f'Unique users:         {df["user_id"].nunique():,}')
print(f'Unique policies:      {df["policy_id"].nunique()}')
print(f'Positive rate:        {df["label"].mean():.2%}')
print(f'Avg interactions/user:{len(df)/df["user_id"].nunique():.1f}')
print()
print('Label distribution:')
print(df['label'].value_counts(normalize=True).map('{:.2%}'.format))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('User Feature Distributions', fontsize=14, fontweight='bold')

# Age distribution
ax = axes[0, 0]
df['age'].hist(bins=30, ax=ax, color='#7c3aed', alpha=0.8, edgecolor='none')
ax.set_title('Age Distribution')
ax.set_xlabel('Age')
ax.set_ylabel('Count')

# Income bracket
ax = axes[0, 1]
df['income_bracket'].value_counts().plot(kind='bar', ax=ax, color='#2563eb', alpha=0.8)
ax.set_title('Income Bracket')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

# City tier
ax = axes[0, 2]
df['city_tier'].value_counts().plot(kind='pie', ax=ax,
    colors=['#7c3aed', '#2563eb', '#38bdf8'], autopct='%1.0f%%')
ax.set_title('City Tier')

# Monthly budget
ax = axes[1, 0]
df['monthly_budget'].hist(bins=40, ax=ax, color='#34d399', alpha=0.8, edgecolor='none')
ax.set_title('Monthly Budget (₹)')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, p: f'₹{x/1000:.0f}K'))

# Policy category distribution
ax = axes[1, 1]
df['policy_category'].value_counts().plot(kind='bar', ax=ax, color='#fbbf24', alpha=0.8)
ax.set_title('Policy Category')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Conversion rate by category
ax = axes[1, 2]
cvr = df.groupby('policy_category')['label'].mean().sort_values(ascending=False)
cvr.plot(kind='bar', ax=ax, color='#fb7185', alpha=0.8)
ax.set_title('Conversion Rate by Category')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('../data/processed/eda_user_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Feature Engineering Validation

In [ ]:
from ml.features.feature_definitions import (
    build_feature_vector, get_feature_names, POLICIES, ALL_FEATURES
)

# Test feature vector on a sample user
sample_user = {
    'user_id': 'eda-001', 'age': 30, 'gender': 'MALE',
    'income_bracket': '6L_TO_10L', 'city_tier': 'TIER_2',
    'is_smoker': False, 'has_diabetes': True, 'has_hypertension': False,
    'has_heart_disease': False, 'family_members': 2,
    'monthly_budget': 2500, 'purchased_categories': ['MOTOR'],
}

print(f'Feature vector length: {len(ALL_FEATURES)}')
print(f'Feature names (first 10): {ALL_FEATURES[:10]}')

# Build feature vectors for all policies
import numpy as np
from ml.training.data_generator import POLICIES

vectors = [build_feature_vector(sample_user, p) for p in POLICIES]
feat_df = pd.DataFrame(vectors, columns=get_feature_names(),
                       index=[p['name'][:25] for p in POLICIES])
print(f'\nFeature matrix shape: {feat_df.shape}')
print(f'Any NaN: {feat_df.isna().any().any()}')
feat_df[['budget_fit_score', 'age_fit_score', 'health_relevance_score',
          'coverage_gap_score', 'income_fit_score']].round(3)

## 3. Training Pipeline Quick Test

In [ ]:
from ml.training.preprocessor import RecommendationPreprocessor

pp = RecommendationPreprocessor()
X = pp.fit_transform(df)
y = df['label'].values

print(f'Feature matrix: {X.shape}')
print(f'Labels: {y.shape} | Positive rate: {y.mean():.2%}')

# Feature correlation with label
import pandas as pd
feat_df_all = pd.DataFrame(X, columns=get_feature_names())
feat_df_all['label'] = y

correlations = feat_df_all.corr()['label'].drop('label').abs().sort_values(ascending=False)
print('\nTop 10 features by correlation with purchase:')
print(correlations.head(10).map('{:.4f}'.format))

In [ ]:
# Feature importance from correlations visualised
fig, ax = plt.subplots(figsize=(10, 6))
correlations.head(15).plot(kind='barh', ax=ax, color='#7c3aed', alpha=0.85)
ax.set_title('Top 15 Feature Correlations with Purchase Label', fontsize=12, fontweight='bold')
ax.set_xlabel('|Pearson Correlation|')
ax.invert_yaxis()
ax.axvline(x=0.05, color='#38bdf8', linestyle='--', alpha=0.7, label='0.05 threshold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/eda_feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Rule-Based vs ML Score Distribution

In [ ]:
from ml.models.recommender import RecommendationEngine
from ml.training.data_generator import POLICIES

engine = RecommendationEngine()

# Score sample user against all policies
scores = engine._rule_based_scores(sample_user, POLICIES)

results = pd.DataFrame({
    'policy': [p['name'][:30] for p in POLICIES],
    'category': [p['category'] for p in POLICIES],
    'rule_score': scores,
    'premium': [p['base_premium'] for p in POLICIES],
}).sort_values('rule_score', ascending=False)

print('Rule-based scores for sample user (age=30, diabetic, family=2, budget=₹2500/mo):')
print(results[['policy', 'category', 'rule_score', 'premium']].to_string(index=False))